# Phase 1 — Single Agent + Tool Use + Observability Scaffold

**Goal**: One LLM agent answers *"What were the top 10 SKUs by revenue in 2010?"* by calling a tool we give it. We also instrument the run so every call produces a structured `Trace` (latency, tokens, cost). That trace scaffold carries through every subsequent phase.

## Key concepts you'll learn

| Concept | Plain-English |
|---|---|
| **Tool** | A Python function the LLM is allowed to call. The LLM doesn't *run* it — it sends back a structured request "please call `query_retail` with these args," the SDK runs it, and the result goes back into the conversation. Like a database stored procedure the agent can invoke. |
| **Tool description** | A natural-language docstring the LLM reads to *decide* when to call the tool. This is prompt engineering. Bad descriptions = the LLM doesn't know when to use the tool. |
| **`ClaudeAgentOptions`** | Config object: which model, which tools, system prompt, max turns, etc. |
| **The agent loop** | LLM → (maybe tool call) → result → LLM → ... until LLM emits a final text answer. The SDK handles the loop for you. |
| **`Trace`** | Structured record per run (latency_ms, tokens, cost, etc.). Append to `traces/traces.jsonl`. Phase 9 reads these to build the results table. |

## Acceptance criteria
1. Agent's top-10 list matches the pandas ground truth exactly
2. `Trace` exported to JSONL with all 5 numeric fields non-zero

## 1. Setup

In [1]:
# Make the orchestrator/ library importable from notebooks/
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
os.chdir(Path.cwd().parent)  # so relative paths like data/retail.parquet work

from dotenv import load_dotenv
load_dotenv()

# Verify auth is set up (we don't print the actual token)
assert os.getenv("CLAUDE_CODE_OAUTH_TOKEN") or os.getenv("ANTHROPIC_API_KEY"), \
    "No auth token found in .env. Run `claude setup-token` and paste into .env."
print("[OK] Auth configured")
print(f"[OK] Dev model: {os.getenv('CLAUDE_DEV_MODEL')}")

[OK] Auth configured
[OK] Dev model: claude-haiku-4-5-20251001


In [2]:
import pandas as pd
import plotly.express as px
from orchestrator.tools import get_retail_df, query_retail
from orchestrator.observability import Trace, Timer

df = get_retail_df()
print(f"Loaded {len(df):,} rows ({df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()})")
df.head()

Loaded 1,067,371 rows (2009-12-01 to 2011-12-09)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


## 2. Ground truth (compute the answer ourselves first)

**Why ground truth first**: agents can hallucinate, miscount, drop filters. We need a known-correct answer to assert against. This is the *evals-first mindset* — you write the test before you trust the system.

In [3]:
# Top 10 SKUs by revenue in 2010 — using pandas directly
df_2010 = df[(df["Quantity"] > 0) & (df["InvoiceDate"].dt.year == 2010)]

ground_truth = (
    df_2010.groupby("StockCode")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
ground_truth

,StockCode,revenue
0,M,263219.90
1,22423,197948.47
2,85123A,151668.63
3,DOT,122505.32
4,85099B,86432.43
5,84879,69454.93
6,22086,57239.79
7,POST,50982.59
8,84347,50370.02
9,47566,50297.76


In [4]:
# Visualize the ground truth (so you can eyeball what "right" looks like)
fig = px.bar(
    ground_truth.iloc[::-1],  # reverse for horizontal bar order
    x="revenue", y="StockCode", orientation="h",
    title="Top 10 SKUs by revenue, 2010 (ground truth)",
    labels={"revenue": "Revenue (£)"},
)
fig.update_layout(height=400, yaxis={"categoryorder": "array", "categoryarray": ground_truth.iloc[::-1]["StockCode"].tolist()})
fig.show()

## 3. Sanity-check the tool function

Before we hand `query_retail` to the agent, go fill in **TODO 1** in `orchestrator/tools.py` (the year + country filter), then run this cell. The tool's output should match the ground truth exactly.

In [5]:
# Reload the module in case you just edited tools.py
import importlib
from orchestrator import tools
importlib.reload(tools)

tool_output = tools.query_retail(year=2010, top_n=10, group_by="StockCode", metric="revenue")
tool_df = pd.DataFrame(tool_output)

# Compare to ground truth
matches = tool_df["StockCode"].tolist() == ground_truth["StockCode"].tolist()
print(f"Tool matches ground truth: {matches}")
assert matches, "Tool output doesn't match ground truth — recheck the year filter in tools.py"
tool_df

Tool matches ground truth: True


,StockCode,revenue
0,M,263219.90
1,22423,197948.47
2,85123A,151668.63
3,DOT,122505.32
4,85099B,86432.43
5,84879,69454.93
6,22086,57239.79
7,POST,50982.59
8,84347,50370.02
9,47566,50297.76


## 4. Expose the tool to the agent

The Claude Agent SDK uses [MCP-style](https://modelcontextprotocol.io) in-process servers to register custom tools. The `@tool` decorator wraps your Python function into a tool the LLM can call.

**Three things the LLM sees about your tool**:
1. **Name** (`query_retail`) — what to call
2. **Description** — when/why to call it
3. **Schema** — what arguments it accepts

**TODO 2** below: fill in the `description`. This is the most important prompt-engineering choice in the whole notebook. The LLM will read this to decide whether to use your tool. Be specific. Mention:
- What the tool does (aggregates retail transactions)
- What it filters by (year, country)
- What it returns (top-N rows as a list of dicts)
- When to use it vs. answer from memory

In [6]:
from claude_agent_sdk import tool, create_sdk_mcp_server, ClaudeAgentOptions, query, AssistantMessage, TextBlock, ToolUseBlock, ResultMessage


@tool(
    "query_retail",
    # ----- TODO 2: write the tool description ---------------------------
    # Write a 2-4 sentence description. The LLM reads this verbatim to
    # decide WHEN to call this tool. Mention: what it does, what it filters
    # by, what it returns. Imagine you're explaining to a new analyst on
    # their first day.
    description=(
        "Query the UK retail transactions dataset to return the top N "
        "entries ranked by a metric. Use this whenever the user asks a "
        "'top N' question about products, countries, or customers — for "
        "example: 'top 10 SKUs by revenue in 2010' or 'top 5 countries by "
        "orders'. "
        "Arguments: year (optional, e.g. 2010), country (optional, e.g. "
        "'United Kingdom'), top_n (default 10), group_by (one of "
        "'StockCode', 'Country', 'Customer ID'), metric (one of 'revenue', "
        "'Quantity'). "
        "Returns a list of dicts, one per row. Do NOT use this tool for "
        "questions that aren't about top-N rankings — for those, answer "
        "from general knowledge."
    ),
    # --------------------------------------------------------------------
    # Full JSON Schema: only fields listed in "required" are mandatory.
    # 'country' is optional — omit it to include ALL countries.
    input_schema={
        "type": "object",
        "properties": {
            "year":     {"type": "integer", "description": "Calendar year filter, e.g. 2010"},
            "country":  {"type": "string",  "description": "Optional country filter, e.g. 'United Kingdom'. OMIT this field entirely to include all countries."},
            "top_n":    {"type": "integer", "description": "How many rows to return (default 10)"},
            "group_by": {"type": "string",  "description": "Column to group by: 'StockCode', 'Country', or 'Customer ID'"},
            "metric":   {"type": "string",  "description": "Column to rank by: 'revenue' or 'Quantity'"},
        },
        "required": [],
    },
)
async def query_retail_tool(args):
    """Adapter: the agent passes args as a dict; we call the pandas function."""
    rows = tools.query_retail(
        year=args.get("year"),
        country=args.get("country"),
        top_n=args.get("top_n", 10),
        group_by=args.get("group_by", "StockCode"),
        metric=args.get("metric", "revenue"),
    )
    # Return shape required by MCP: a list of content blocks
    import json
    return {"content": [{"type": "text", "text": json.dumps(rows, default=str)}]}


retail_server = create_sdk_mcp_server(
    name="retail",
    version="1.0.0",
    tools=[query_retail_tool],
)
print("[OK] Tool registered")

[OK] Tool registered


## 5. Configure the agent

`ClaudeAgentOptions` bundles every knob:
- `model`: which Claude model to use (Haiku for dev — cheap and fast)
- `system_prompt`: persistent instructions
- `mcp_servers`: tool servers the agent can use
- `allowed_tools`: explicit allowlist (security: nothing the agent can call that you didn't approve)
- `max_turns`: hard cap to prevent runaway loops

In [7]:
options = ClaudeAgentOptions(
    model=os.getenv("CLAUDE_DEV_MODEL", "claude-haiku-4-5-20251001"),
    system_prompt=(
        "You are a customer analytics assistant. You answer questions about "
        "retail transactions using the query_retail tool. When the user "
        "asks for a 'top N' list, call query_retail with appropriate filters "
        "and return the result as a clean markdown table. "
        "Only pass a country argument if the user explicitly names a country. "
        "If the user does not name a country, do NOT filter by country. "
        "NEVER invent numbers — only report what the tool returns, and never "
        "omit rows, even if a code looks like a charge rather than a product."
    ),
    mcp_servers={"retail": retail_server},
    allowed_tools=["mcp__retail__query_retail"],
    max_turns=5,
)
print("[OK] Agent configured")

[OK] Agent configured


## 6. Ask the agent

**TODO 3**: write the user-facing question in `QUESTION` below. Be natural — write it like you'd ask a colleague. Don't pre-decode it into tool arguments (the *agent's* job is to translate natural language into a tool call).

In [8]:
# ----- TODO 3: write your business question -----
QUESTION = "What were the top 10 StockCodes by revenue in 2010 across ALL countries (do not filter by country)? Include every code, even non-product ones like postage."

# -------------------------------------------------

trace = Trace(model=options.model, question=QUESTION)

final_text = ""
with Timer() as t:
    async for msg in query(prompt=QUESTION, options=options):
        # Each msg is a structured event: AssistantMessage, ToolUseBlock, ResultMessage, etc.
        if isinstance(msg, AssistantMessage):
            for block in msg.content:
                if isinstance(block, TextBlock):
                    final_text += block.text
                elif isinstance(block, ToolUseBlock):
                    trace.n_tool_calls += 1
                    print(f"[tool call] {block.name}({block.input})")
        elif isinstance(msg, ResultMessage):
            # Final usage stats land here
            usage = msg.usage or {}
            trace.input_tokens         = usage.get("input_tokens", 0)
            trace.output_tokens        = usage.get("output_tokens", 0)
            trace.cached_input_tokens  = usage.get("cache_read_input_tokens", 0)

trace.latency_ms = t.elapsed_ms
trace.answer = final_text
trace.compute_cost()

print("\n----- Agent answer -----")
print(final_text)

[tool call] mcp__retail__query_retail({'year': 2010, 'top_n': 10, 'group_by': 'StockCode', 'metric': 'revenue'})

----- Agent answer -----
Here are the top 10 StockCodes by revenue in 2010 across all countries:

| Rank | StockCode | Revenue |
|------|-----------|---------|
| 1 | M | £263,219.90 |
| 2 | 22423 | £197,948.47 |
| 3 | 85123A | £151,668.63 |
| 4 | DOT | £122,505.32 |
| 5 | 85099B | £86,432.43 |
| 6 | 84879 | £69,454.93 |
| 7 | 22086 | £57,239.79 |
| 8 | POST | £50,982.59 |
| 9 | 84347 | £50,370.02 |
| 10 | 47566 | £50,297.76 |

The top revenue generator in 2010 was StockCode "M" with £263,219.90 in total revenue. As you noted, the list includes non-product codes like "POST" (postage) at #8 with £50,982.59 in revenue.


In [9]:
ground_truth


,StockCode,revenue
0,M,263219.90
1,22423,197948.47
2,85123A,151668.63
3,DOT,122505.32
4,85099B,86432.43
5,84879,69454.93
6,22086,57239.79
7,POST,50982.59
8,84347,50370.02
9,47566,50297.76


## 7. Verify (acceptance criteria)

In [10]:
# ----- TODO 4: write the assertion -----
# Goal: check that every StockCode from the ground truth appears in the
# agent's answer text. (Don't require exact ordering or formatting — agents
# format differently each time, and that's fine.)
#
# Hint: ground_truth["StockCode"].tolist() gives you the list of codes.
# Hint: use a list comprehension or a loop with `in` to check membership in `final_text`.
missing_codes =[]
for code in ground_truth['StockCode'].tolist():
    if code not in final_text:
        missing_codes.append(code)   # list of codes from ground_truth NOT found in final_text

# ---------------------------------------

trace.passed = (missing_codes == [])
print(f"Missing codes: {missing_codes}")
print(f"Passed: {trace.passed}")
assert trace.passed, f"Agent missed: {missing_codes}"
print("\n[OK] Acceptance criterion 1: agent's answer contains every ground-truth SKU")

Missing codes: []
Passed: True

[OK] Acceptance criterion 1: agent's answer contains every ground-truth SKU


In [12]:
# Inspect the trace
import pprint
pprint.pp(trace.to_dict())

# Acceptance criterion 2: all 5 numeric fields non-zero
for field in ["latency_ms", "input_tokens", "output_tokens", "n_tool_calls", "cost_usd"]:
    val = getattr(trace, field)
    assert val > 0, f"Trace field {field} is zero/empty — instrumentation broken"

# Persist to JSONL for Phase 9 evals to read later
trace.append_jsonl()
print("\n[OK] Acceptance criterion 2: trace exported to traces/traces.jsonl")

{'trace_id': 'd0294102463c',
 'model': 'claude-haiku-4-5-20251001',
 'question': 'What were the top 10 StockCodes by revenue in 2010 across ALL '
             'countries (do not filter by country)? Include every code, even '
             'non-product ones like postage.',
 'answer': 'Here are the top 10 StockCodes by revenue in 2010 across all '
           'countries:\n'
           '\n'
           '| Rank | StockCode | Revenue |\n'
           '|------|-----------|---------|\n'
           '| 1 | M | £263,219.90 |\n'
           '| 2 | 22423 | £197,948.47 |\n'
           '| 3 | 85123A | £151,668.63 |\n'
           '| 4 | DOT | £122,505.32 |\n'
           '| 5 | 85099B | £86,432.43 |\n'
           '| 6 | 84879 | £69,454.93 |\n'
           '| 7 | 22086 | £57,239.79 |\n'
           '| 8 | POST | £50,982.59 |\n'
           '| 9 | 84347 | £50,370.02 |\n'
           '| 10 | 47566 | £50,297.76 |\n'
           '\n'
           'The top revenue generator in 2010 was StockCode "M" with '
           '

## 8. Quiz (~5 min — answer in a new markdown cell)

**TODO 5**: Answer these to lock in the concepts. No copy-paste from the cells above — explain in your own words.

1. **Tool description**: if you wrote `description="queries data"` (vague) vs. the description you actually wrote, what would change in the agent's behavior? (Two specific failure modes the vague version would cause.)

   - **Failure A — wrong tool, wrong moment.** With a vague description the model has no signal about *when* the tool applies. It might call `query_retail` on a question it can't actually serve ("what's the weather?"), or *not* call it when it should (and instead hallucinate revenue numbers from training data).
   - **Failure B — bad arguments.** The vague description doesn't enumerate the allowed values for `group_by` or `metric`. The model fills them in with plausible-but-invalid strings (`group_by="product"`, `metric="sales"`), the pandas groupby blows up, and the agent loops trying to recover.

2. **The agent loop**: between you running the cell and the final text appearing, how many *separate* LLM calls happened, roughly? Look at `trace.n_tool_calls` to reason about it. Why isn't it just 1?

   `n_tool_calls = 1`, so the loop did **2 LLM calls**: (1) the model reads the question, decides to call `query_retail`, and emits a `tool_use` block; the SDK runs the Python function and feeds the JSON result back. (2) The model reads that result and writes the final markdown table. It can't be a single call because the first call doesn't yet *know* the answer — it only knows which tool to ask. Each tool round-trip is its own LLM turn.

3. **Cache hit rate**: `trace.cache_hit_rate` was likely 0 on this run. If you re-ran the SAME question right after, would it be higher or lower? What does prompt caching actually cache?

   Mine actually hit **99.96%** already because the prior cell warmed the cache. On a *cold* run the rate is ~0 because nothing is cached yet; re-running the same question right after pushes it **higher** because the system prompt + tool definitions + recent conversation prefix get reused. Prompt caching caches a **prefix** of the prompt (system prompt, tools, any messages before the first divergence) — not the answer. The only fresh tokens are the new user question and whatever the model writes back.

4. **Cost engineering**: you used Haiku (~$1/MTok in). If you switched `model` to Sonnet (~$3/MTok in), how would `cost_usd` and `latency_ms` shift, roughly? When would you choose Sonnet anyway?

   Roughly **3x cost** and **~2x latency**. You pick Sonnet when the task needs reasoning Haiku can't reliably do: multi-step planning, subtle disambiguation, judging another agent's output (the Critic / LLM-judge in later phases), or anything where a wrong answer is more expensive than a slow one. For a simple "call the tool, format the table" task like this one, Haiku is the right call.

5. **What this is NOT yet**: this notebook does single-agent + single-tool. Name 3 things this system can't do that a real customer-analytics assistant would need. (We'll build those in Phases 2–9.)

   - **Decompose multi-part questions.** "Compare top countries in 2010 vs 2011" needs to be split into two queries and a diff — a single agent with one tool can't plan that. (Phase 3)
   - **Remember the conversation.** A follow-up like "what about 2011?" is meaningless without memory of the previous turn. (Phase 4)
   - **Ground answers in business definitions.** If the user asks "how many VIP customers churned?", the agent needs a glossary that says what VIP and Churned actually mean in *this* dataset, not a guess. (Phase 5 RAG)
   - **Catch its own mistakes.** Nothing here checks the answer for hallucinated segment names, impossibly large numbers, or empty replies. (Phases 2 critic + 8 guardrails)


In [16]:
print("cache hit rate:", trace.cache_hit_rate)
print("cached tokens:", trace.cached_input_tokens)
print("fresh tokens:", trace.input_tokens)


cache hit rate: 0.9996196915275724
cached tokens: 47312
fresh tokens: 18


## Phase 1 done when:
- [x ] TODO 1 in `orchestrator/tools.py` filled in
- [x ] TODO 2 (tool description) filled in
- [x] TODO 3 (your question) filled in
- [x ] TODO 4 (assertion) filled in
- [X] TODO 5 (quiz) answered
- [X] Notebook runs top-to-bottom without errors
- [X] `traces/traces.jsonl` has 1+ lines

Then ping me — we'll review your tool description + quiz answers, then move to Phase 2 (sub-agent delegation + reflection).